In [3]:
import os

print("=== /kaggle/input ===")
for item in os.listdir("/kaggle/input"):
    print(item)

print("\n=== ALL FILES ===")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

=== /kaggle/input ===
datasets

=== ALL FILES ===
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_124.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_949.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_786.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_371.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_599.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_802.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_1323.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_1347.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_955.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_778.jpg
/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training/pituitary/Tr-pi_1354.jpg
/kaggle/input/datasets/ekrasafdar/

In [4]:
import os
os.system('find /kaggle/input -iname "*pillar1*" -o -iname "*.keras"')

/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss
/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss/pillar1_anatomical_plausibility_loss.py
/kaggle/input/datasets/ekrasafdar/model-mobilenetv2-seed42-keras/model_mobilenetv2_seed42.keras


0

In [5]:
import os
d = '/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss-py'
print("exists:", os.path.exists(d))
if os.path.exists(d):
    for root, dirs, files in os.walk(d):
        for f in files:
            print(os.path.join(root, f))

exists: False


In [6]:
os.system('find /kaggle/input -iname "*pillar1*"')

/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss
/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss/pillar1_anatomical_plausibility_loss.py


0

In [7]:
# ============================================================
# Cell 2 — Load data + checkpoint, compute anatomical masks
# ============================================================
import sys
sys.path.append('/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss')

import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image
import os, glob

from pillar1_anatomical_plausibility_loss import (
    build_anatomical_mask, precompute_masks, APLModel,
    watermark_masking_augment, compute_edge_bias_reduction
)

DATA = "/kaggle/input/datasets/ekrasafdar/brain-tumor-mri"
CHECKPOINT_PATH = "/kaggle/input/datasets/ekrasafdar/model-mobilenetv2-seed42-keras/model_mobilenetv2_seed42.keras"
IMG_SIZE = (224, 224)
CLASS_ORDER = ['glioma', 'meningioma', 'notumor', 'pituitary']

def load_split(split_dir):
    images, labels = [], []
    for class_idx, class_name in enumerate(CLASS_ORDER):
        pattern = os.path.join(split_dir, class_name, "*")
        for fpath in glob.glob(pattern):
            try:
                img = Image.open(fpath).convert("RGB").resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(class_idx)
            except Exception as e:
                print(f"skipped {fpath}: {e}")
    images = np.stack(images, axis=0)
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=4)
    return images, labels_onehot

print("Loading training images (this takes a few minutes)...")
X_train_raw, y_train = load_split(f"{DATA}/Training")
print(f"Train: {X_train_raw.shape}")

print("Loading test images...")
X_test_raw, y_test = load_split(f"{DATA}/Testing")
print(f"Test: {X_test_raw.shape}")

n_val = int(0.10 * len(X_train_raw))
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train_raw))
val_idx, train_idx = idx[:n_val], idx[n_val:]
X_val_raw, y_val = X_train_raw[val_idx], y_train[val_idx]
X_train_raw, y_train = X_train_raw[train_idx], y_train[train_idx]
print(f"After carving validation: train={len(X_train_raw)}, val={len(X_val_raw)}")

print("Computing anatomical masks (Otsu thresholding)...")
train_masks = precompute_masks(X_train_raw)
val_masks = precompute_masks(X_val_raw)
test_masks = precompute_masks(X_test_raw)
print("Masks done:", train_masks.shape, val_masks.shape, test_masks.shape)

print("Loading baseline MobileNetV2 checkpoint...")
base_model = tf.keras.models.load_model(CHECKPOINT_PATH, compile=False)
print("Checkpoint loaded:", base_model.count_params(), "params")

Loading training images (this takes a few minutes)...
Train: (5600, 224, 224, 3)
Loading test images...
Test: (1600, 224, 224, 3)
After carving validation: train=5040, val=560
Computing anatomical masks (Otsu thresholding)...
Masks done: (5040, 224, 224) (560, 224, 224) (1600, 224, 224)
Loading baseline MobileNetV2 checkpoint...


I0000 00:00:1789986580.304934      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789986580.308248      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Checkpoint loaded: 2263108 params


In [8]:
# ============================================================
# Cell 2.5 — Patch: fix the tf.cond bug in watermark_masking_augment
# ============================================================
import tensorflow as tf

def watermark_masking_augment(image, p=0.5, border_frac=0.12, seed=None):
    rng = tf.random.Generator.from_seed(seed) if seed is not None else tf.random.get_global_generator()
    do_it = rng.uniform([], 0, 1) < p

    h = tf.shape(image)[0]
    w = tf.shape(image)[1]
    bh = tf.cast(tf.cast(h, tf.float32) * border_frac, tf.int32)
    bw = tf.cast(tf.cast(w, tf.float32) * border_frac, tf.int32)
    mean_val = tf.reduce_mean(image)

    def apply_mask():
        region = rng.uniform([], 0, 4, dtype=tf.int32)
        base_mask = tf.ones_like(image)

        def zero_region(y0, y1, x0, x1, m):
            idx_y = tf.range(h)[:, None]
            idx_x = tf.range(w)[None, :]
            in_y = tf.logical_and(idx_y >= y0, idx_y < y1)
            in_x = tf.logical_and(idx_x >= x0, idx_x < x1)
            in_region = tf.logical_and(in_y, in_x)
            in_region = tf.cast(in_region, tf.float32)[..., None]
            return m * (1.0 - in_region)

        final_mask = tf.case([
            (tf.equal(region, 0), lambda: zero_region(0, bh, 0, w, base_mask)),
            (tf.equal(region, 1), lambda: zero_region(0, bh, w - bw, w, base_mask)),
            (tf.equal(region, 2), lambda: zero_region(h - bh, h, 0, bw, base_mask)),
            (tf.equal(region, 3), lambda: zero_region(h - bh, h, 0, w, base_mask)),
        ])
        return image * final_mask + (1.0 - final_mask) * mean_val

    def no_mask():
        return image

    return tf.cond(do_it, apply_mask, no_mask)

print("Patched watermark_masking_augment for graph-mode compatibility.")

Patched watermark_masking_augment for graph-mode compatibility.


In [9]:
# ============================================================
# Cell 2.6 — Patch: fix float64/float32 mismatch in focal_loss
# ============================================================
import tensorflow as tf

def focal_loss_fixed(self, y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    ce = -y_true * tf.math.log(y_pred)
    weight = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
    return tf.reduce_sum(weight * ce, axis=-1)

APLModel.focal_loss = focal_loss_fixed
print("Patched focal_loss for dtype compatibility.")

Patched focal_loss for dtype compatibility.


In [10]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [11]:
# ============================================================
# Cell 3 — APL fine-tuning (the actual Pillar 1 training step)
# ============================================================
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Fix dtype mismatch (numpy defaults to float64, model expects float32)
y_train = y_train.astype('float32')
y_val = y_val.astype('float32')

def make_apl_dataset(images_uint8, masks, labels_onehot, batch_size=32, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, masks, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)

    def _map(img, mask, lbl):
        img = tf.cast(img, tf.float32)
        if augment:
            img = watermark_masking_augment(img, p=0.5)
        img = preprocess_input(img)
        return (img, mask), lbl

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_apl_dataset(X_train_raw, train_masks, y_train, batch_size=32, shuffle=True, augment=True)
val_ds = make_apl_dataset(X_val_raw, val_masks, y_val, batch_size=32, shuffle=False, augment=False)

LAMBDA_APL = 0.15

apl_model = APLModel(base_model, lambda_apl=LAMBDA_APL)
apl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")],
)

print(f"Starting APL fine-tune, lambda={LAMBDA_APL}...")
history = apl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_acc", patience=5, restore_best_weights=True)],
)

apl_model.base_model.save("/kaggle/working/mobilenetv2_apl_seed42.keras")
print("Saved APL-trained model.")

Starting APL fine-tune, lambda=0.15...
Epoch 1/12


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:670: UserWarning: `model.compiled_metrics()` is deprecated. Instead, use e.g.:
```
for metric in self.metrics:
    metric.update_state(y, y_pred)
```

  return self._compiled_metrics_update_state(
2026-09-21 10:30:18.042377: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-21 10:30:18.201474: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - apl_penalty: 49.3678 - acc: 0.7768 - focal_loss: 0.9267 - loss: 8.3319 - val_loss: 0.5265
Epoch 5/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - apl_penalty: 112.8600 - acc: 0.6161 - focal_loss: 1.1857 - loss: 18.1147 - val_loss: 1.1566
Epoch 6/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 99ms/step - apl_penalty: 4.4991 - acc: 0.4946 - focal_loss: 1.8007 - loss: 2.4756 - val_loss: 1.3881
Epoch 7/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 100ms/step - apl_penalty: 0.3775 - acc: 0.4643 - focal_loss: 0.9544 - loss: 1.0110 - val_loss: 1.1110
Epoch 8/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 100ms/step - apl_penalty: 0.1148 - acc: 0.4786 - focal_loss: 0.8866 - loss: 0.9039 - val_loss: 0.9795
Epoch 9/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 99ms/step - apl_penalty: 0.6415 - acc: 0.4750 - focal_loss: 1.9803 - loss: 2.0765 - val_loss: 0.8237
Epoch 10/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - apl_penalty: 3.9757 - acc: 0.4696 - focal_loss: 1.4597 - loss: 2.0561 - val_lo

In [12]:
import tensorflow as tf

def train_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)

    with tf.GradientTape() as outer_tape:
        with tf.GradientTape() as saliency_tape:
            saliency_tape.watch(images)
            y_pred = self.base_model(images, training=True)
            pred_class_idx = tf.argmax(y_true, axis=-1)
            batch_idx = tf.range(tf.shape(y_pred)[0], dtype=tf.int64)
            gather_idx = tf.stack([batch_idx, tf.cast(pred_class_idx, tf.int64)], axis=1)
            class_scores = tf.gather_nd(y_pred, gather_idx)

        saliency = saliency_tape.gradient(class_scores, images)
        saliency = tf.reduce_sum(tf.abs(saliency), axis=-1)

        off_target = 1.0 - masks
        off_target_mass = tf.reduce_sum(saliency * off_target, axis=[1, 2])
        total_mass = tf.reduce_sum(saliency, axis=[1, 2]) + 1e-8
        apl_penalty = tf.reduce_mean(off_target_mass / total_mass)  # normalized ratio, bounded [0,1] — matches edge-bias definition

        focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
        total_loss = focal + self.lambda_apl * apl_penalty

    grads = outer_tape.gradient(total_loss, self.base_model.trainable_variables)
    self.optimizer.apply_gradients(zip(grads, self.base_model.trainable_variables))

    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results.update({"loss": total_loss, "focal_loss": focal, "apl_penalty": apl_penalty})
    return results

def test_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)
    y_pred = self.base_model(images, training=False)
    focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results["loss"] = focal
    return results

APLModel.train_step = train_step_fixed
APLModel.test_step = test_step_fixed
print("Patched: apl_penalty is now a bounded [0,1] ratio, not an unbounded raw sum.")

Patched: apl_penalty is now a bounded [0,1] ratio, not an unbounded raw sum.


In [13]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

y_train = y_train.astype('float32')
y_val = y_val.astype('float32')

def make_apl_dataset(images_uint8, masks, labels_onehot, batch_size=32, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, masks, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)

    def _map(img, mask, lbl):
        img = tf.cast(img, tf.float32)
        if augment:
            img = watermark_masking_augment(img, p=0.5)
        img = preprocess_input(img)
        return (img, mask), lbl

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_apl_dataset(X_train_raw, train_masks, y_train, batch_size=32, shuffle=True, augment=True)
val_ds = make_apl_dataset(X_val_raw, val_masks, y_val, batch_size=32, shuffle=False, augment=False)

LAMBDA_APL = 0.15

apl_model = APLModel(base_model, lambda_apl=LAMBDA_APL)
apl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")],
)

print(f"Starting APL fine-tune, lambda={LAMBDA_APL}...")
history = apl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)],
)

apl_model.base_model.save("/kaggle/working/mobilenetv2_apl_seed42.keras")
print("Saved APL-trained model (best-weights-restored, not just last epoch).")

Starting APL fine-tune, lambda=0.15...
Epoch 1/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 68s 241ms/step - apl_penalty: 0.2381 - acc: 0.8429 - focal_loss: 0.0349 - loss: 0.0706 - val_loss: 0.2412
Epoch 2/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 101ms/step - apl_penalty: 0.2881 - acc: 0.8661 - focal_loss: 0.0269 - loss: 0.0701 - val_loss: 0.1882
Epoch 3/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 101ms/step - apl_penalty: 0.2527 - acc: 0.8714 - focal_loss: 0.0083 - loss: 0.0462 - val_loss: 0.0427
Epoch 4/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 100ms/step - apl_penalty: 0.3178 - acc: 0.9196 - focal_loss: 0.0107 - loss: 0.0583 - val_loss: 0.0239
Epoch 5/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 100ms/step - apl_penalty: 0.2526 - acc: 0.9357 - focal_loss: 0.0071 - loss: 0.0450 - val_loss: 0.0185
Epoch 6/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 100ms/step - apl_penalty: 0.2564 - acc: 0.9607 - focal_loss: 0.0034 - loss: 0.0419 - val_loss: 0.0054
Epoch 7/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 100ms/step - apl_penalty: 0.2133 - acc: 0

In [14]:
# Sanity check: does this model's val performance transfer to the real, separate test set?
X_test_prep = preprocess_input(tf.cast(X_test_raw, tf.float32))
test_acc = (apl_model.base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
print(f"APL-trained model test accuracy: {test_acc:.4f}")

APL-trained model test accuracy: 0.9275


In [15]:
import sys
sys.path.append('/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss')

import os, glob, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image

from pillar1_anatomical_plausibility_loss import precompute_masks

DATA = "/kaggle/input/datasets/ekrasafdar/brain-tumor-mri"
CHECKPOINT_PATH = "/kaggle/input/datasets/ekrasafdar/model-mobilenetv2-seed42-keras/model_mobilenetv2_seed42.keras"
APL_MODEL_PATH = "/kaggle/working/mobilenetv2_apl_seed42.keras"
IMG_SIZE = (224, 224)
CLASS_ORDER = ['glioma', 'meningioma', 'notumor', 'pituitary']
BASELINE_EDGE_BIAS_NOTUMOR = 0.1340

def load_split(split_dir):
    images, labels = [], []
    for class_idx, class_name in enumerate(CLASS_ORDER):
        pattern = os.path.join(split_dir, class_name, "*")
        for fpath in glob.glob(pattern):
            try:
                img = Image.open(fpath).convert("RGB").resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(class_idx)
            except Exception as e:
                print(f"skipped {fpath}: {e}")
    images = np.stack(images, axis=0)
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=4)
    return images, labels_onehot

print("Reloading test images...")
X_test_raw, y_test = load_split(f"{DATA}/Testing")
print(f"Test: {X_test_raw.shape}")

print("Computing anatomical masks for test set...")
test_masks = precompute_masks(X_test_raw)

print("Checking for APL-trained model on disk...")
if not os.path.exists(APL_MODEL_PATH):
    raise FileNotFoundError(
        f"{APL_MODEL_PATH} not found -- the APL training was lost with the session restart. "
        f"You need to re-run the training cell (Cell 3) again before this will work."
    )

print("Loading baseline checkpoint...")
base_model = tf.keras.models.load_model(CHECKPOINT_PATH, compile=False)

print("Loading APL-trained model...")
apl_base_model = tf.keras.models.load_model(APL_MODEL_PATH, compile=False)

def compute_saliency_and_edgebias(model, images_uint8, masks, batch_size=32):
    n = len(images_uint8)
    all_edge_bias = []
    for i in range(0, n, batch_size):
        batch_imgs = images_uint8[i:i+batch_size]
        batch_masks = masks[i:i+batch_size]
        imgs_prep = preprocess_input(tf.cast(batch_imgs, tf.float32))
        with tf.GradientTape() as tape:
            tape.watch(imgs_prep)
            out = model(imgs_prep, training=False)
            pred_idx = tf.argmax(out, axis=-1)
            gathered = tf.gather_nd(out, tf.stack(
                [tf.range(tf.shape(out)[0], dtype=tf.int64), pred_idx], axis=1))
        grads = tape.gradient(gathered, imgs_prep)
        saliency = tf.reduce_sum(tf.abs(grads), axis=-1).numpy()
        off_target_mass = (saliency * (1.0 - batch_masks)).sum(axis=(1, 2))
        total_mass = saliency.sum(axis=(1, 2)) + 1e-8
        all_edge_bias.append(off_target_mass / total_mass)
    return np.concatenate(all_edge_bias)

notumor_idx = 2
notumor_mask_bool = np.argmax(y_test, axis=1) == notumor_idx
X_notumor = X_test_raw[notumor_mask_bool]
masks_notumor = test_masks[notumor_mask_bool]

print("\n=== BEFORE (baseline model) ===")
eb_before = compute_saliency_and_edgebias(base_model, X_notumor, masks_notumor)
print(f"Mean edge-bias (no-tumor class): {eb_before.mean():.4f}  (paper baseline: {BASELINE_EDGE_BIAS_NOTUMOR:.4f})")

print("\n=== AFTER (APL fine-tuned model) ===")
eb_after = compute_saliency_and_edgebias(apl_base_model, X_notumor, masks_notumor)
print(f"Mean edge-bias (no-tumor class): {eb_after.mean():.4f}")

reduction_pct = 100.0 * (eb_before.mean() - eb_after.mean()) / eb_before.mean()
print(f"\nRelative reduction: {reduction_pct:+.1f}%")

X_test_prep = preprocess_input(tf.cast(X_test_raw, tf.float32))
acc_before = (base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
acc_after = (apl_base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
print(f"\nTest accuracy: baseline={acc_before:.4f}, APL-trained={acc_after:.4f}")

results = {
    "edge_bias_before": float(eb_before.mean()),
    "edge_bias_after": float(eb_after.mean()),
    "relative_reduction_pct": float(reduction_pct),
    "accuracy_before": float(acc_before),
    "accuracy_after": float(acc_after),
}
with open("/kaggle/working/pillar1_results_mobilenetv2.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved results.")
print(results)

Reloading test images...
Test: (1600, 224, 224, 3)
Computing anatomical masks for test set...
Checking for APL-trained model on disk...
Loading baseline checkpoint...
Loading APL-trained model...

=== BEFORE (baseline model) ===
Mean edge-bias (no-tumor class): 0.3562  (paper baseline: 0.1340)

=== AFTER (APL fine-tuned model) ===
Mean edge-bias (no-tumor class): 0.2086

Relative reduction: +41.4%

Test accuracy: baseline=0.8912, APL-trained=0.9275

Saved results.
{'edge_bias_before': 0.35621392726898193, 'edge_bias_after': 0.20857277512550354, 'relative_reduction_pct': 41.44732666015625, 'accuracy_before': 0.89125, 'accuracy_after': 0.9275}


In [ ]:
# ============================================================
# COMPLETE PILLAR 1 PIPELINE — ONE CELL, NO DEPENDENCY ON PRIOR STATE
# ============================================================
import sys
sys.path.append('/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss')

import os, glob, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image

from pillar1_anatomical_plausibility_loss import precompute_masks, APLModel

DATA = "/kaggle/input/datasets/ekrasafdar/brain-tumor-mri"
CHECKPOINT_PATH = "/kaggle/input/datasets/ekrasafdar/model-mobilenetv2-seed42-keras/model_mobilenetv2_seed42.keras"
IMG_SIZE = (224, 224)
CLASS_ORDER = ['glioma', 'meningioma', 'notumor', 'pituitary']
BASELINE_EDGE_BIAS_NOTUMOR = 0.1340
LAMBDA_APL = 0.15

# ---------- Load data ----------
def load_split(split_dir):
    images, labels = [], []
    for class_idx, class_name in enumerate(CLASS_ORDER):
        pattern = os.path.join(split_dir, class_name, "*")
        for fpath in glob.glob(pattern):
            try:
                img = Image.open(fpath).convert("RGB").resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(class_idx)
            except Exception as e:
                print(f"skipped {fpath}: {e}")
    images = np.stack(images, axis=0)
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=4)
    return images, labels_onehot

print("Loading training images...")
X_train_raw, y_train = load_split(f"{DATA}/Training")
print("Loading test images...")
X_test_raw, y_test = load_split(f"{DATA}/Testing")

n_val = int(0.10 * len(X_train_raw))
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train_raw))
val_idx, train_idx = idx[:n_val], idx[n_val:]
X_val_raw, y_val = X_train_raw[val_idx], y_train[val_idx]
X_train_raw, y_train = X_train_raw[train_idx], y_train[train_idx]
y_train = y_train.astype('float32')
y_val = y_val.astype('float32')
print(f"train={len(X_train_raw)}, val={len(X_val_raw)}, test={len(X_test_raw)}")

print("Computing anatomical masks...")
train_masks = precompute_masks(X_train_raw)
val_masks = precompute_masks(X_val_raw)
test_masks = precompute_masks(X_test_raw)

print("Loading baseline checkpoint...")
base_model = tf.keras.models.load_model(CHECKPOINT_PATH, compile=False)

# ---------- Patch 1: graph-mode-safe watermark augmentation ----------
def watermark_masking_augment(image, p=0.5, border_frac=0.12, seed=None):
    rng_ = tf.random.get_global_generator()
    do_it = rng_.uniform([], 0, 1) < p
    h = tf.shape(image)[0]
    w = tf.shape(image)[1]
    bh = tf.cast(tf.cast(h, tf.float32) * border_frac, tf.int32)
    bw = tf.cast(tf.cast(w, tf.float32) * border_frac, tf.int32)
    mean_val = tf.reduce_mean(image)

    def apply_mask():
        region = rng_.uniform([], 0, 4, dtype=tf.int32)
        base_mask = tf.ones_like(image)
        def zero_region(y0, y1, x0, x1, m):
            idx_y = tf.range(h)[:, None]
            idx_x = tf.range(w)[None, :]
            in_y = tf.logical_and(idx_y >= y0, idx_y < y1)
            in_x = tf.logical_and(idx_x >= x0, idx_x < x1)
            in_region = tf.cast(tf.logical_and(in_y, in_x), tf.float32)[..., None]
            return m * (1.0 - in_region)
        final_mask = tf.case([
            (tf.equal(region, 0), lambda: zero_region(0, bh, 0, w, base_mask)),
            (tf.equal(region, 1), lambda: zero_region(0, bh, w - bw, w, base_mask)),
            (tf.equal(region, 2), lambda: zero_region(h - bh, h, 0, bw, base_mask)),
            (tf.equal(region, 3), lambda: zero_region(h - bh, h, 0, w, base_mask)),
        ])
        return image * final_mask + (1.0 - final_mask) * mean_val

    return tf.cond(do_it, apply_mask, lambda: image)

# ---------- Patch 2: dtype-safe focal loss ----------
def focal_loss_fixed(self, y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    ce = -y_true * tf.math.log(y_pred)
    weight = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
    return tf.reduce_sum(weight * ce, axis=-1)

# ---------- Patch 3: normalized (bounded) APL penalty ----------
def train_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)
    with tf.GradientTape() as outer_tape:
        with tf.GradientTape() as saliency_tape:
            saliency_tape.watch(images)
            y_pred = self.base_model(images, training=True)
            pred_class_idx = tf.argmax(y_true, axis=-1)
            batch_idx = tf.range(tf.shape(y_pred)[0], dtype=tf.int64)
            gather_idx = tf.stack([batch_idx, tf.cast(pred_class_idx, tf.int64)], axis=1)
            class_scores = tf.gather_nd(y_pred, gather_idx)
        saliency = saliency_tape.gradient(class_scores, images)
        saliency = tf.reduce_sum(tf.abs(saliency), axis=-1)
        off_target = 1.0 - masks
        off_target_mass = tf.reduce_sum(saliency * off_target, axis=[1, 2])
        total_mass = tf.reduce_sum(saliency, axis=[1, 2]) + 1e-8
        apl_penalty = tf.reduce_mean(off_target_mass / total_mass)
        focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
        total_loss = focal + self.lambda_apl * apl_penalty
    grads = outer_tape.gradient(total_loss, self.base_model.trainable_variables)
    self.optimizer.apply_gradients(zip(grads, self.base_model.trainable_variables))
    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results.update({"loss": total_loss, "focal_loss": focal, "apl_penalty": apl_penalty})
    return results

def test_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)
    y_pred = self.base_model(images, training=False)
    focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results["loss"] = focal
    return results

APLModel.focal_loss = focal_loss_fixed
APLModel.train_step = train_step_fixed
APLModel.test_step = test_step_fixed
print("All patches applied.")

# ---------- Build datasets ----------
def make_apl_dataset(images_uint8, masks, labels_onehot, batch_size=32, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, masks, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)
    def _map(img, mask, lbl):
        img = tf.cast(img, tf.float32)
        if augment:
            img = watermark_masking_augment(img, p=0.5)
        img = preprocess_input(img)
        return (img, mask), lbl
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_apl_dataset(X_train_raw, train_masks, y_train, batch_size=32, shuffle=True, augment=True)
val_ds = make_apl_dataset(X_val_raw, val_masks, y_val, batch_size=32, shuffle=False, augment=False)

# ---------- Train ----------
apl_model = APLModel(base_model, lambda_apl=LAMBDA_APL)
apl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")],
)
print(f"\nStarting APL fine-tune, lambda={LAMBDA_APL}...")
apl_model.fit(
    train_ds, validation_data=val_ds, epochs=12,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)],
)
apl_model.base_model.save("/kaggle/working/mobilenetv2_apl_seed42.keras")
print("Saved APL-trained model.")

# ---------- Evaluate: edge-bias before/after ----------
def compute_saliency_and_edgebias(model, images_uint8, masks, batch_size=32):
    n = len(images_uint8)
    all_edge_bias = []
    for i in range(0, n, batch_size):
        batch_imgs = images_uint8[i:i+batch_size]
        batch_masks = masks[i:i+batch_size]
        imgs_prep = preprocess_input(tf.cast(batch_imgs, tf.float32))
        with tf.GradientTape() as tape:
            tape.watch(imgs_prep)
            out = model(imgs_prep, training=False)
            pred_idx = tf.argmax(out, axis=-1)
            gathered = tf.gather_nd(out, tf.stack(
                [tf.range(tf.shape(out)[0], dtype=tf.int64), pred_idx], axis=1))
        grads = tape.gradient(gathered, imgs_prep)
        saliency = tf.reduce_sum(tf.abs(grads), axis=-1).numpy()
        off_target_mass = (saliency * (1.0 - batch_masks)).sum(axis=(1, 2))
        total_mass = saliency.sum(axis=(1, 2)) + 1e-8
        all_edge_bias.append(off_target_mass / total_mass)
    return np.concatenate(all_edge_bias)

notumor_idx = 2
notumor_mask_bool = np.argmax(y_test, axis=1) == notumor_idx
X_notumor = X_test_raw[notumor_mask_bool]
masks_notumor = test_masks[notumor_mask_bool]

print("\n=== BEFORE (baseline) ===")
eb_before = compute_saliency_and_edgebias(base_model, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_before.mean():.4f} (paper baseline: {BASELINE_EDGE_BIAS_NOTUMOR:.4f})")

print("\n=== AFTER (APL-trained) ===")
eb_after = compute_saliency_and_edgebias(apl_model.base_model, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_after.mean():.4f}")

reduction_pct = 100.0 * (eb_before.mean() - eb_after.mean()) / eb_before.mean()
print(f"\nRelative reduction: {reduction_pct:+.1f}%")

X_test_prep = preprocess_input(tf.cast(X_test_raw, tf.float32))
acc_before = (base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
acc_after = (apl_model.base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
print(f"\nAccuracy: baseline={acc_before:.4f}, APL-trained={acc_after:.4f}")

results = {
    "lambda_apl": LAMBDA_APL,
    "edge_bias_before": float(eb_before.mean()),
    "edge_bias_after": float(eb_after.mean()),
    "relative_reduction_pct": float(reduction_pct),
    "accuracy_before": float(acc_before),
    "accuracy_after": float(acc_after),
}
with open("/kaggle/working/pillar1_results_mobilenetv2.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n=== FINAL RESULTS ===")
print(results)

Loading training images...
Loading test images...
train=5040, val=560, test=1600
Computing anatomical masks...
Loading baseline checkpoint...
All patches applied.

Starting APL fine-tune, lambda=0.15...
Epoch 1/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 66s 237ms/step - apl_penalty: 0.3559 - acc: 0.9446 - focal_loss: 0.0151 - loss: 0.0685 - val_loss: 0.0439


In [ ]:
# ==================== COMPLETE PILLAR 1 PIPELINE — ONE CELL ====================
import sys
sys.path.append('/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss')

import os, glob, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image

from pillar1_anatomical_plausibility_loss import (
    build_anatomical_mask, precompute_masks, APLModel,
    watermark_masking_augment, compute_edge_bias_reduction
)

DATA = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
CHECKPOINT_PATH = '/kaggle/input/datasets/ekrasafdar/model-mobilenetv2-seed42-keras/model_mobilenetv2_seed42.keras'
IMG_SIZE = (224, 224)
CLASS_ORDER = ['glioma', 'meningioma', 'notumor', 'pituitary']
BASELINE_EDGE_BIAS_NOTUMOR = 0.1348
LAMBDA_APL = 0.15

# ---------- Load data ----------
def load_split(split_dir):
    images, labels = [], []
    for class_idx, class_name in enumerate(CLASS_ORDER):
        pattern = os.path.join(split_dir, class_name, "*")
        for fpath in glob.glob(pattern):
            try:
                img = Image.open(fpath).convert("RGB").resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(class_idx)
            except Exception as e:
                print(f"skipped {fpath}: {e}")
    images = np.stack(images, axis=0)
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=4)
    return images, labels_onehot

print("Loading training images...")
X_train_raw, y_train = load_split(f"{DATA}/Training")
print("Loading test images...")
X_test_raw, y_test = load_split(f"{DATA}/Testing")

n_val = int(0.10 * len(X_train_raw))
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train_raw))
val_idx, train_idx = idx[:n_val], idx[n_val:]
X_val_raw, y_val = X_train_raw[val_idx], y_train[val_idx]
X_train_raw, y_train = X_train_raw[train_idx], y_train[train_idx]
y_train = y_train.astype('float32')
y_val = y_val.astype('float32')

print("Computing anatomical masks...")
train_masks = precompute_masks(X_train_raw)
val_masks = precompute_masks(X_val_raw)
test_masks = precompute_masks(X_test_raw)

print("Loading baseline checkpoint...")
base_model = tf.keras.models.load_model(CHECKPOINT_PATH, compile=False)

# ---------- Patch 2: dtype-safe focal loss ----------
def focal_loss_fixed(self, y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    ce = -y_true * tf.math.log(y_pred)
    weight = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
    return tf.reduce_sum(weight * ce, axis=-1)

APLModel.focal_loss = focal_loss_fixed

# ---------- Build datasets ----------
def make_apl_dataset(images, masks, labels, batch_size=32, shuffle=True, augment=True):
    ds = tf.data.Dataset.from_tensor_slices((images, masks, labels))
    if shuffle:
        ds = ds.shuffle(len(images), seed=42)
    if augment:
        ds = ds.map(lambda img, m, y: (watermark_masking_augment(img, p=0.5), m, y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda img, m, y: (preprocess_input(tf.cast(img, tf.float32)), m, y),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda img, m, y: ((img, m), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_apl_dataset(X_train_raw, train_masks, y_train, shuffle=True, augment=True)
val_ds = make_apl_dataset(X_val_raw, val_masks, y_val, shuffle=False, augment=False)

# ---------- FIX: clone base_model so training doesn't mutate the baseline ----------
apl_train_model = tf.keras.models.clone_model(base_model)
apl_train_model.set_weights(base_model.get_weights())

apl_model = APLModel(apl_train_model, lambda_apl=LAMBDA_APL)
apl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")],
)

print(f"Starting APL fine-tune, lambda={LAMBDA_APL}...")
history = apl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)],
)

apl_model.base_model.save("/kaggle/working/mobilenetv2_apl_seed42.keras")
print("Saved APL-trained model.")

# ---------- Eval: BEFORE (untouched baseline) vs AFTER (trained clone) ----------
notumor_idx = np.where(np.argmax(y_test, axis=1) == CLASS_ORDER.index('notumor'))[0]
X_notumor = X_test_raw[notumor_idx]
masks_notumor = test_masks[notumor_idx]

print("\n=== BEFORE (baseline) ===")
eb_before = compute_edge_bias_reduction(base_model, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_before.mean():.4f} (paper baseline: {BASELINE_EDGE_BIAS_NOTUMOR:.4f})")

print("\n=== AFTER (APL-trained) ===")
eb_after = compute_edge_bias_reduction(apl_model.base_model, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_after.mean():.4f}")

reduction_pct = 100.0 * (eb_before.mean() - eb_after.mean()) / eb_before.mean()
print(f"\nRelative reduction: {reduction_pct:+.1f}%")

X_test_prep = preprocess_input(tf.cast(X_test_raw, tf.float32))
acc_before = (base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
acc_after = (apl_model.base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
print(f"\nAccuracy: baseline={acc_before:.4f}, APL-trained={acc_after:.4f}")

results = {
    "lambda_apl": LAMBDA_APL,
    "edge_bias_before": float(eb_before.mean()),
    "edge_bias_after": float(eb_after.mean()),
    "relative_reduction_pct": float(reduction_pct),
    "accuracy_before": float(acc_before),
    "accuracy_after": float(acc_after),
}
with open("/kaggle/working/pillar1_results_mobilenetv2.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n=== FINAL RESULTS ===")
print(results)

In [1]:
# ============================================================
# COMPLETE PILLAR 1 PIPELINE — ONE CELL, NO DEPENDENCY ON PRIOR STATE
# ============================================================
import sys
sys.path.append('/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss')

import os, glob, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image

from pillar1_anatomical_plausibility_loss import precompute_masks, APLModel

DATA = "/kaggle/input/datasets/ekrasafdar/brain-tumor-mri"
CHECKPOINT_PATH = "/kaggle/input/datasets/ekrasafdar/model-mobilenetv2-seed42-keras/model_mobilenetv2_seed42.keras"
IMG_SIZE = (224, 224)
CLASS_ORDER = ['glioma', 'meningioma', 'notumor', 'pituitary']
BASELINE_EDGE_BIAS_NOTUMOR = 0.1340
LAMBDA_APL = 0.15

# ---------- Load data ----------
def load_split(split_dir):
    images, labels = [], []
    for class_idx, class_name in enumerate(CLASS_ORDER):
        pattern = os.path.join(split_dir, class_name, "*")
        for fpath in glob.glob(pattern):
            try:
                img = Image.open(fpath).convert("RGB").resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(class_idx)
            except Exception as e:
                print(f"skipped {fpath}: {e}")
    images = np.stack(images, axis=0)
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=4)
    return images, labels_onehot

print("Loading training images...")
X_train_raw, y_train = load_split(f"{DATA}/Training")
print("Loading test images...")
X_test_raw, y_test = load_split(f"{DATA}/Testing")

n_val = int(0.10 * len(X_train_raw))
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train_raw))
val_idx, train_idx = idx[:n_val], idx[n_val:]
X_val_raw, y_val = X_train_raw[val_idx], y_train[val_idx]
X_train_raw, y_train = X_train_raw[train_idx], y_train[train_idx]
y_train = y_train.astype('float32')
y_val = y_val.astype('float32')
print(f"train={len(X_train_raw)}, val={len(X_val_raw)}, test={len(X_test_raw)}")

print("Computing anatomical masks...")
train_masks = precompute_masks(X_train_raw)
val_masks = precompute_masks(X_val_raw)
test_masks = precompute_masks(X_test_raw)

print("Loading baseline checkpoint...")
base_model = tf.keras.models.load_model(CHECKPOINT_PATH, compile=False)

# ---------- Patch 1: graph-mode-safe watermark augmentation ----------
def watermark_masking_augment(image, p=0.5, border_frac=0.12, seed=None):
    rng_ = tf.random.get_global_generator()
    do_it = rng_.uniform([], 0, 1) < p
    h = tf.shape(image)[0]
    w = tf.shape(image)[1]
    bh = tf.cast(tf.cast(h, tf.float32) * border_frac, tf.int32)
    bw = tf.cast(tf.cast(w, tf.float32) * border_frac, tf.int32)
    mean_val = tf.reduce_mean(image)

    def apply_mask():
        region = rng_.uniform([], 0, 4, dtype=tf.int32)
        base_mask = tf.ones_like(image)
        def zero_region(y0, y1, x0, x1, m):
            idx_y = tf.range(h)[:, None]
            idx_x = tf.range(w)[None, :]
            in_y = tf.logical_and(idx_y >= y0, idx_y < y1)
            in_x = tf.logical_and(idx_x >= x0, idx_x < x1)
            in_region = tf.cast(tf.logical_and(in_y, in_x), tf.float32)[..., None]
            return m * (1.0 - in_region)
        final_mask = tf.case([
            (tf.equal(region, 0), lambda: zero_region(0, bh, 0, w, base_mask)),
            (tf.equal(region, 1), lambda: zero_region(0, bh, w - bw, w, base_mask)),
            (tf.equal(region, 2), lambda: zero_region(h - bh, h, 0, bw, base_mask)),
            (tf.equal(region, 3), lambda: zero_region(h - bh, h, 0, w, base_mask)),
        ])
        return image * final_mask + (1.0 - final_mask) * mean_val

    return tf.cond(do_it, apply_mask, lambda: image)

# ---------- Patch 2: dtype-safe focal loss ----------
def focal_loss_fixed(self, y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    ce = -y_true * tf.math.log(y_pred)
    weight = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
    return tf.reduce_sum(weight * ce, axis=-1)

# ---------- Patch 3: normalized (bounded) APL penalty ----------
def train_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)
    with tf.GradientTape() as outer_tape:
        with tf.GradientTape() as saliency_tape:
            saliency_tape.watch(images)
            y_pred = self.base_model(images, training=True)
            pred_class_idx = tf.argmax(y_true, axis=-1)
            batch_idx = tf.range(tf.shape(y_pred)[0], dtype=tf.int64)
            gather_idx = tf.stack([batch_idx, tf.cast(pred_class_idx, tf.int64)], axis=1)
            class_scores = tf.gather_nd(y_pred, gather_idx)
        saliency = saliency_tape.gradient(class_scores, images)
        saliency = tf.reduce_sum(tf.abs(saliency), axis=-1)
        off_target = 1.0 - masks
        off_target_mass = tf.reduce_sum(saliency * off_target, axis=[1, 2])
        total_mass = tf.reduce_sum(saliency, axis=[1, 2]) + 1e-8
        apl_penalty = tf.reduce_mean(off_target_mass / total_mass)
        focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
        total_loss = focal + self.lambda_apl * apl_penalty
    grads = outer_tape.gradient(total_loss, self.base_model.trainable_variables)
    self.optimizer.apply_gradients(zip(grads, self.base_model.trainable_variables))
    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results.update({"loss": total_loss, "focal_loss": focal, "apl_penalty": apl_penalty})
    return results

def test_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)
    y_pred = self.base_model(images, training=False)
    focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results["loss"] = focal
    return results

APLModel.focal_loss = focal_loss_fixed
APLModel.train_step = train_step_fixed
APLModel.test_step = test_step_fixed
print("All patches applied.")

# ---------- Build datasets ----------
def make_apl_dataset(images_uint8, masks, labels_onehot, batch_size=32, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, masks, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)
    def _map(img, mask, lbl):
        img = tf.cast(img, tf.float32)
        if augment:
            img = watermark_masking_augment(img, p=0.5)
        img = preprocess_input(img)
        return (img, mask), lbl
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_apl_dataset(X_train_raw, train_masks, y_train, batch_size=32, shuffle=True, augment=True)
val_ds = make_apl_dataset(X_val_raw, val_masks, y_val, batch_size=32, shuffle=False, augment=False)

# ---------- Train ----------
apl_model = APLModel(base_model, lambda_apl=LAMBDA_APL)
apl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")],
)
print(f"\nStarting APL fine-tune, lambda={LAMBDA_APL}...")
apl_model.fit(
    train_ds, validation_data=val_ds, epochs=12,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)],
)
apl_model.base_model.save("/kaggle/working/mobilenetv2_apl_seed42.keras")
print("Saved APL-trained model.")

# ---------- Evaluate: edge-bias before/after ----------
def compute_saliency_and_edgebias(model, images_uint8, masks, batch_size=32):
    n = len(images_uint8)
    all_edge_bias = []
    for i in range(0, n, batch_size):
        batch_imgs = images_uint8[i:i+batch_size]
        batch_masks = masks[i:i+batch_size]
        imgs_prep = preprocess_input(tf.cast(batch_imgs, tf.float32))
        with tf.GradientTape() as tape:
            tape.watch(imgs_prep)
            out = model(imgs_prep, training=False)
            pred_idx = tf.argmax(out, axis=-1)
            gathered = tf.gather_nd(out, tf.stack(
                [tf.range(tf.shape(out)[0], dtype=tf.int64), pred_idx], axis=1))
        grads = tape.gradient(gathered, imgs_prep)
        saliency = tf.reduce_sum(tf.abs(grads), axis=-1).numpy()
        off_target_mass = (saliency * (1.0 - batch_masks)).sum(axis=(1, 2))
        total_mass = saliency.sum(axis=(1, 2)) + 1e-8
        all_edge_bias.append(off_target_mass / total_mass)
    return np.concatenate(all_edge_bias)

notumor_idx = 2
notumor_mask_bool = np.argmax(y_test, axis=1) == notumor_idx
X_notumor = X_test_raw[notumor_mask_bool]
masks_notumor = test_masks[notumor_mask_bool]

print("\n=== BEFORE (baseline) ===")
eb_before = compute_saliency_and_edgebias(base_model, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_before.mean():.4f} (paper baseline: {BASELINE_EDGE_BIAS_NOTUMOR:.4f})")

print("\n=== AFTER (APL-trained) ===")
eb_after = compute_saliency_and_edgebias(apl_model.base_model, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_after.mean():.4f}")

reduction_pct = 100.0 * (eb_before.mean() - eb_after.mean()) / eb_before.mean()
print(f"\nRelative reduction: {reduction_pct:+.1f}%")

X_test_prep = preprocess_input(tf.cast(X_test_raw, tf.float32))
acc_before = (base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
acc_after = (apl_model.base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
print(f"\nAccuracy: baseline={acc_before:.4f}, APL-trained={acc_after:.4f}")

results = {
    "lambda_apl": LAMBDA_APL,
    "edge_bias_before": float(eb_before.mean()),
    "edge_bias_after": float(eb_after.mean()),
    "relative_reduction_pct": float(reduction_pct),
    "accuracy_before": float(acc_before),
    "accuracy_after": float(acc_after),
}
with open("/kaggle/working/pillar1_results_mobilenetv2.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n=== FINAL RESULTS ===")
print(results)

Loading training images...
Loading test images...
train=5040, val=560, test=1600
Computing anatomical masks...
Loading baseline checkpoint...


I0000 00:00:1790151416.930574      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790151416.933412      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


All patches applied.

Starting APL fine-tune, lambda=0.15...
Epoch 1/12


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:670: UserWarning: `model.compiled_metrics()` is deprecated. Instead, use e.g.:
```
for metric in self.metrics:
    metric.update_state(y, y_pred)
```

  return self._compiled_metrics_update_state(
2026-09-23 08:17:33.847456: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 08:17:33.985056: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 08:17:38.470067: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 08:17:3

157/158 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - apl_penalty: 0.3849 - acc: 0.9335 - focal_loss: 0.0201 - loss: 0.0778

2026-09-23 08:18:15.379004: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 08:18:15.516104: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


158/158 ━━━━━━━━━━━━━━━━━━━━ 95s 315ms/step - apl_penalty: 0.3877 - acc: 0.9446 - focal_loss: 0.0031 - loss: 0.0612 - val_loss: 0.0179
Epoch 2/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 96ms/step - apl_penalty: 0.3902 - acc: 0.9696 - focal_loss: 0.0207 - loss: 0.0793 - val_loss: 0.0024
Epoch 3/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 96ms/step - apl_penalty: 0.3585 - acc: 0.9464 - focal_loss: 4.4991e-04 - loss: 0.0542 - val_loss: 0.0259
Epoch 4/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - apl_penalty: 0.3372 - acc: 0.9679 - focal_loss: 0.0343 - loss: 0.0848 - val_loss: 1.6608e-04
Epoch 5/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - apl_penalty: 0.3010 - acc: 0.9696 - focal_loss: 9.9564e-04 - loss: 0.0461 - val_loss: 4.7023e-04
Epoch 6/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - apl_penalty: 0.2822 - acc: 0.9625 - focal_loss: 0.0323 - loss: 0.0746 - val_loss: 8.1543e-07
Epoch 7/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 97ms/step - apl_penalty: 0.3453 - acc: 0.9625 - focal_loss: 7.9037e-04 - los

In [3]:
# ============================================================
# COMPLETE PILLAR 1 PIPELINE — FIXED (baseline snapshot bug resolved)
# ============================================================
import sys
sys.path.append('/kaggle/input/datasets/ekrasafdar/pillar1-anatomical-plausibility-loss')

import os, glob, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image

from pillar1_anatomical_plausibility_loss import precompute_masks, APLModel

DATA = "/kaggle/input/datasets/ekrasafdar/brain-tumor-mri"
CHECKPOINT_PATH = "/kaggle/input/datasets/ekrasafdar/model-mobilenetv2-seed42-keras/model_mobilenetv2_seed42.keras"
IMG_SIZE = (224, 224)
CLASS_ORDER = ['glioma', 'meningioma', 'notumor', 'pituitary']
BASELINE_EDGE_BIAS_NOTUMOR = 0.1340  # paper's Grad-CAM-based figure -- NOTE: this script uses
                                       # input-gradient saliency, a related but different metric,
                                       # so don't report the two side-by-side as directly equivalent
LAMBDA_APL = 0.15

# ---------- Load data ----------
def load_split(split_dir):
    images, labels = [], []
    for class_idx, class_name in enumerate(CLASS_ORDER):
        pattern = os.path.join(split_dir, class_name, "*")
        for fpath in glob.glob(pattern):
            try:
                img = Image.open(fpath).convert("RGB").resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(class_idx)
            except Exception as e:
                print(f"skipped {fpath}: {e}")
    images = np.stack(images, axis=0)
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=4)
    return images, labels_onehot

print("Loading training images...")
X_train_raw, y_train = load_split(f"{DATA}/Training")
print("Loading test images...")
X_test_raw, y_test = load_split(f"{DATA}/Testing")

n_val = int(0.10 * len(X_train_raw))
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train_raw))
val_idx, train_idx = idx[:n_val], idx[n_val:]
X_val_raw, y_val = X_train_raw[val_idx], y_train[val_idx]
X_train_raw, y_train = X_train_raw[train_idx], y_train[train_idx]
y_train = y_train.astype('float32')
y_val = y_val.astype('float32')
print(f"train={len(X_train_raw)}, val={len(X_val_raw)}, test={len(X_test_raw)}")

print("Computing anatomical masks...")
train_masks = precompute_masks(X_train_raw)
val_masks = precompute_masks(X_val_raw)
test_masks = precompute_masks(X_test_raw)

print("Loading baseline checkpoint...")
base_model = tf.keras.models.load_model(CHECKPOINT_PATH, compile=False)

# ---------- CRITICAL FIX: snapshot the baseline to disk BEFORE training touches it ----------
# APLModel stores base_model by reference, so training mutates it in place.
# Without this snapshot, "before" and "after" would be the same object.
BASELINE_SNAPSHOT_PATH = "/kaggle/working/baseline_snapshot.keras"
base_model.save(BASELINE_SNAPSHOT_PATH)
print(f"Saved untouched baseline snapshot to {BASELINE_SNAPSHOT_PATH}")

# ---------- Patch 1: graph-mode-safe watermark augmentation ----------
def watermark_masking_augment(image, p=0.5, border_frac=0.12, seed=None):
    rng_ = tf.random.get_global_generator()
    do_it = rng_.uniform([], 0, 1) < p
    h = tf.shape(image)[0]
    w = tf.shape(image)[1]
    bh = tf.cast(tf.cast(h, tf.float32) * border_frac, tf.int32)
    bw = tf.cast(tf.cast(w, tf.float32) * border_frac, tf.int32)
    mean_val = tf.reduce_mean(image)

    def apply_mask():
        region = rng_.uniform([], 0, 4, dtype=tf.int32)
        base_mask = tf.ones_like(image)
        def zero_region(y0, y1, x0, x1, m):
            idx_y = tf.range(h)[:, None]
            idx_x = tf.range(w)[None, :]
            in_y = tf.logical_and(idx_y >= y0, idx_y < y1)
            in_x = tf.logical_and(idx_x >= x0, idx_x < x1)
            in_region = tf.cast(tf.logical_and(in_y, in_x), tf.float32)[..., None]
            return m * (1.0 - in_region)
        final_mask = tf.case([
            (tf.equal(region, 0), lambda: zero_region(0, bh, 0, w, base_mask)),
            (tf.equal(region, 1), lambda: zero_region(0, bh, w - bw, w, base_mask)),
            (tf.equal(region, 2), lambda: zero_region(h - bh, h, 0, bw, base_mask)),
            (tf.equal(region, 3), lambda: zero_region(h - bh, h, 0, w, base_mask)),
        ])
        return image * final_mask + (1.0 - final_mask) * mean_val

    return tf.cond(do_it, apply_mask, lambda: image)

# ---------- Patch 2: dtype-safe focal loss ----------
def focal_loss_fixed(self, y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    ce = -y_true * tf.math.log(y_pred)
    weight = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
    return tf.reduce_sum(weight * ce, axis=-1)

# ---------- Patch 3: normalized (bounded [0,1]) APL penalty ----------
def train_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)
    with tf.GradientTape() as outer_tape:
        with tf.GradientTape() as saliency_tape:
            saliency_tape.watch(images)
            y_pred = self.base_model(images, training=True)
            pred_class_idx = tf.argmax(y_true, axis=-1)
            batch_idx = tf.range(tf.shape(y_pred)[0], dtype=tf.int64)
            gather_idx = tf.stack([batch_idx, tf.cast(pred_class_idx, tf.int64)], axis=1)
            class_scores = tf.gather_nd(y_pred, gather_idx)
        saliency = saliency_tape.gradient(class_scores, images)
        saliency = tf.reduce_sum(tf.abs(saliency), axis=-1)
        off_target = 1.0 - masks
        off_target_mass = tf.reduce_sum(saliency * off_target, axis=[1, 2])
        total_mass = tf.reduce_sum(saliency, axis=[1, 2]) + 1e-8
        apl_penalty = tf.reduce_mean(off_target_mass / total_mass)
        focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
        total_loss = focal + self.lambda_apl * apl_penalty
    grads = outer_tape.gradient(total_loss, self.base_model.trainable_variables)
    self.optimizer.apply_gradients(zip(grads, self.base_model.trainable_variables))
    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results.update({"loss": total_loss, "focal_loss": focal, "apl_penalty": apl_penalty})
    return results

def test_step_fixed(self, data):
    (images, masks), y_true = data
    y_true = tf.cast(y_true, tf.float32)
    y_pred = self.base_model(images, training=False)
    focal = tf.reduce_mean(self.focal_loss(y_true, y_pred))
    self.compiled_metrics.update_state(y_true, y_pred)
    results = {m.name: m.result() for m in self.metrics}
    results["loss"] = focal
    return results

APLModel.focal_loss = focal_loss_fixed
APLModel.train_step = train_step_fixed
APLModel.test_step = test_step_fixed
print("All patches applied.")

# ---------- Build datasets ----------
def make_apl_dataset(images_uint8, masks, labels_onehot, batch_size=32, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, masks, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)
    def _map(img, mask, lbl):
        img = tf.cast(img, tf.float32)
        if augment:
            img = watermark_masking_augment(img, p=0.5)
        img = preprocess_input(img)
        return (img, mask), lbl
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_apl_dataset(X_train_raw, train_masks, y_train, batch_size=32, shuffle=True, augment=True)
val_ds = make_apl_dataset(X_val_raw, val_masks, y_val, batch_size=32, shuffle=False, augment=False)

# ---------- Train ----------
apl_model = APLModel(base_model, lambda_apl=LAMBDA_APL)  # this DOES mutate base_model in place -- that's why the snapshot above exists
apl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")],
)
print(f"\nStarting APL fine-tune, lambda={LAMBDA_APL}...")
apl_model.fit(
    train_ds, validation_data=val_ds, epochs=12,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)],
)
apl_model.base_model.save("/kaggle/working/mobilenetv2_apl_seed42.keras")
print("Saved APL-trained model.")

# ---------- Evaluate: edge-bias before/after ----------
def compute_saliency_and_edgebias(model, images_uint8, masks, batch_size=32):
    n = len(images_uint8)
    all_edge_bias = []
    for i in range(0, n, batch_size):
        batch_imgs = images_uint8[i:i+batch_size]
        batch_masks = masks[i:i+batch_size]
        imgs_prep = preprocess_input(tf.cast(batch_imgs, tf.float32))
        with tf.GradientTape() as tape:
            tape.watch(imgs_prep)
            out = model(imgs_prep, training=False)
            pred_idx = tf.argmax(out, axis=-1)
            gathered = tf.gather_nd(out, tf.stack(
                [tf.range(tf.shape(out)[0], dtype=tf.int64), pred_idx], axis=1))
        grads = tape.gradient(gathered, imgs_prep)
        saliency = tf.reduce_sum(tf.abs(grads), axis=-1).numpy()
        off_target_mass = (saliency * (1.0 - batch_masks)).sum(axis=(1, 2))
        total_mass = saliency.sum(axis=(1, 2)) + 1e-8
        all_edge_bias.append(off_target_mass / total_mass)
    return np.concatenate(all_edge_bias)

# Reload the UNTOUCHED baseline from disk -- do NOT reuse the `base_model` variable,
# it has been mutated by training since it's the same object apl_model wraps.
baseline_for_eval = tf.keras.models.load_model(BASELINE_SNAPSHOT_PATH, compile=False)

notumor_idx = 2
notumor_mask_bool = np.argmax(y_test, axis=1) == notumor_idx
X_notumor = X_test_raw[notumor_mask_bool]
masks_notumor = test_masks[notumor_mask_bool]

print("\n=== BEFORE (true untouched baseline) ===")
eb_before = compute_saliency_and_edgebias(baseline_for_eval, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_before.mean():.4f} (paper's Grad-CAM baseline: {BASELINE_EDGE_BIAS_NOTUMOR:.4f}, different metric -- see note above)")

print("\n=== AFTER (APL-trained) ===")
eb_after = compute_saliency_and_edgebias(apl_model.base_model, X_notumor, masks_notumor)
print(f"Edge-bias: {eb_after.mean():.4f}")

reduction_pct = 100.0 * (eb_before.mean() - eb_after.mean()) / eb_before.mean()
print(f"\nRelative reduction: {reduction_pct:+.1f}%")

X_test_prep = preprocess_input(tf.cast(X_test_raw, tf.float32))
acc_before = (baseline_for_eval.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
acc_after = (apl_model.base_model.predict(X_test_prep, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
print(f"\nAccuracy: baseline={acc_before:.4f}, APL-trained={acc_after:.4f}")

# Sanity check -- this should now print False
print(f"\nSanity check (before==after must be False): {np.isclose(eb_before.mean(), eb_after.mean())}")

results = {
    "lambda_apl": LAMBDA_APL,
    "edge_bias_before": float(eb_before.mean()),
    "edge_bias_after": float(eb_after.mean()),
    "relative_reduction_pct": float(reduction_pct),
    "accuracy_before": float(acc_before),
    "accuracy_after": float(acc_after),
}
with open("/kaggle/working/pillar1_results_mobilenetv2.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n=== FINAL RESULTS ===")
print(results)

Loading training images...
Loading test images...
train=5040, val=560, test=1600
Computing anatomical masks...
Loading baseline checkpoint...
Saved untouched baseline snapshot to /kaggle/working/baseline_snapshot.keras
All patches applied.

Starting APL fine-tune, lambda=0.15...
Epoch 1/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 69s 246ms/step - apl_penalty: 0.4004 - acc: 0.9536 - focal_loss: 0.0179 - loss: 0.0780 - val_loss: 0.0507
Epoch 2/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - apl_penalty: 0.4058 - acc: 0.9464 - focal_loss: 0.0029 - loss: 0.0638 - val_loss: 0.0053
Epoch 3/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 98ms/step - apl_penalty: 0.3537 - acc: 0.9625 - focal_loss: 0.0118 - loss: 0.0649 - val_loss: 0.0456
Epoch 4/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 97ms/step - apl_penalty: 0.3670 - acc: 0.9571 - focal_loss: 0.0012 - loss: 0.0562 - val_loss: 0.0093
Epoch 5/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - apl_penalty: 0.3721 - acc: 0.9518 - focal_loss: 0.0325 - loss: 0.0884 - val_loss: 0

In [4]:
# ============================================================
# ResNet50 — baseline training + APL fine-tune (reuses loaded data/masks)
# ============================================================
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as prep_rn50
from tensorflow.keras import layers, models

# ---------- Train baseline ----------
def make_plain_dataset(images_uint8, masks, labels_onehot, prep_fn, batch_size=32, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)
    def _map(img, lbl):
        img = prep_fn(tf.cast(img, tf.float32))
        return img, lbl
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_plain = make_plain_dataset(X_train_raw, train_masks, y_train, prep_rn50)
val_ds_plain = make_plain_dataset(X_val_raw, val_masks, y_val, prep_rn50, shuffle=False)

backbone = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
backbone.trainable = False
rn50_model = models.Sequential([
    backbone, layers.GlobalAveragePooling2D(), layers.Dropout(0.4), layers.Dense(4, activation="softmax")
])
rn50_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
print("Training ResNet50 baseline (frozen backbone)...")
rn50_model.fit(train_ds_plain, validation_data=val_ds_plain, epochs=20,
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

backbone.trainable = True
for l in backbone.layers[:-30]:
    l.trainable = False
rn50_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
print("Fine-tuning ResNet50 (last 30 layers unfrozen)...")
rn50_model.fit(train_ds_plain, validation_data=val_ds_plain, epochs=10,
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

rn50_baseline_path = "/kaggle/working/resnet50_baseline_seed42.keras"
rn50_model.save(rn50_baseline_path)
print(f"Saved ResNet50 baseline to {rn50_baseline_path}")

# ---------- APL fine-tune on ResNet50 ----------
rn50_baseline = tf.keras.models.load_model(rn50_baseline_path, compile=False)
rn50_snapshot_path = "/kaggle/working/resnet50_baseline_snapshot.keras"
rn50_baseline.save(rn50_snapshot_path)  # untouched copy for fair before/after

def make_apl_dataset_rn50(images_uint8, masks, labels_onehot, batch_size=32, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, masks, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)
    def _map(img, mask, lbl):
        img = tf.cast(img, tf.float32)
        if augment:
            img = watermark_masking_augment(img, p=0.5)
        img = prep_rn50(img)
        return (img, mask), lbl
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_apl = make_apl_dataset_rn50(X_train_raw, train_masks, y_train, augment=True)
val_ds_apl = make_apl_dataset_rn50(X_val_raw, val_masks, y_val, shuffle=False, augment=False)

rn50_apl_model = APLModel(rn50_baseline, lambda_apl=0.15)
rn50_apl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")])
print("\nStarting ResNet50 APL fine-tune...")
rn50_apl_model.fit(train_ds_apl, validation_data=val_ds_apl, epochs=12,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)])
rn50_apl_model.base_model.save("/kaggle/working/resnet50_apl_seed42.keras")

# ---------- Evaluate ----------
rn50_baseline_eval = tf.keras.models.load_model(rn50_snapshot_path, compile=False)

def compute_saliency_and_edgebias_prep(model, images_uint8, masks, prep_fn, batch_size=32):
    n = len(images_uint8)
    all_eb = []
    for i in range(0, n, batch_size):
        bi, bm = images_uint8[i:i+batch_size], masks[i:i+batch_size]
        imgs_prep = prep_fn(tf.cast(bi, tf.float32))
        with tf.GradientTape() as tape:
            tape.watch(imgs_prep)
            out = model(imgs_prep, training=False)
            pred_idx = tf.argmax(out, axis=-1)
            gathered = tf.gather_nd(out, tf.stack([tf.range(tf.shape(out)[0], dtype=tf.int64), pred_idx], axis=1))
        grads = tape.gradient(gathered, imgs_prep)
        saliency = tf.reduce_sum(tf.abs(grads), axis=-1).numpy()
        off_mass = (saliency * (1.0 - bm)).sum(axis=(1, 2))
        total_mass = saliency.sum(axis=(1, 2)) + 1e-8
        all_eb.append(off_mass / total_mass)
    return np.concatenate(all_eb)

notumor_idx = 2
notumor_mask_bool = np.argmax(y_test, axis=1) == notumor_idx
X_notumor = X_test_raw[notumor_mask_bool]
masks_notumor = test_masks[notumor_mask_bool]

eb_before = compute_saliency_and_edgebias_prep(rn50_baseline_eval, X_notumor, masks_notumor, prep_rn50)
eb_after = compute_saliency_and_edgebias_prep(rn50_apl_model.base_model, X_notumor, masks_notumor, prep_rn50)
reduction_pct = 100.0 * (eb_before.mean() - eb_after.mean()) / eb_before.mean()

X_test_prep_rn50 = prep_rn50(tf.cast(X_test_raw, tf.float32))
acc_before = (rn50_baseline_eval.predict(X_test_prep_rn50, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
acc_after = (rn50_apl_model.base_model.predict(X_test_prep_rn50, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()

print(f"\nSanity check (must be False): {np.isclose(eb_before.mean(), eb_after.mean())}")
results_rn50 = {
    "architecture": "ResNet50", "lambda_apl": 0.15,
    "edge_bias_before": float(eb_before.mean()), "edge_bias_after": float(eb_after.mean()),
    "relative_reduction_pct": float(reduction_pct),
    "accuracy_before": float(acc_before), "accuracy_after": float(acc_after),
}
with open("/kaggle/working/pillar1_results_resnet50.json", "w") as f:
    json.dump(results_rn50, f, indent=2)
print("\n=== RESNET50 FINAL RESULTS ===")
print(results_rn50)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training ResNet50 baseline (frozen backbone)...
Epoch 1/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 39s 165ms/step - accuracy: 0.7427 - loss: 0.6720 - val_accuracy: 0.8839 - val_loss: 0.2964
Epoch 2/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 92ms/step - accuracy: 0.8534 - loss: 0.3793 - val_accuracy: 0.9071 - val_loss: 0.2522
Epoch 3/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 93ms/step - accuracy: 0.8784 - loss: 0.3193 - val_accuracy: 0.9125 - val_loss: 0.2230
Epoch 4/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 92ms/step - accuracy: 0.8998 - loss: 0.2717 - val_accuracy: 0.9161 - val_loss: 0.2103
Epoch 5/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.9028 - loss: 0.2540 - val_accuracy: 0.9286 - val_loss: 0.1852
Epoch 6/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.9097 - loss: 0.2454 - val_accuracy: 0.9339 - val_loss: 0.1823
Epoch 7/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.9153 - loss: 0.2313 - val_accuracy: 0.9339 - val_los

2026-09-23 08:54:26.627244: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 08:54:26.774850: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


157/158 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step - apl_penalty: 0.4366 - acc: 0.9620 - focal_loss: 0.0208 - loss: 0.0863

2026-09-23 08:55:30.427282: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 08:55:30.570634: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


158/158 ━━━━━━━━━━━━━━━━━━━━ 109s 456ms/step - apl_penalty: 0.4766 - acc: 0.9429 - focal_loss: 0.0196 - loss: 0.0911 - val_loss: 0.0740
Epoch 2/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 50s 317ms/step - apl_penalty: 0.4238 - acc: 0.9571 - focal_loss: 0.0222 - loss: 0.0857 - val_loss: 0.0397
Epoch 3/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 50s 314ms/step - apl_penalty: 0.4390 - acc: 0.9714 - focal_loss: 0.0012 - loss: 0.0670 - val_loss: 7.8416e-05
Epoch 4/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 50s 315ms/step - apl_penalty: 0.3404 - acc: 0.9750 - focal_loss: 0.0014 - loss: 0.0525 - val_loss: 0.0054
Epoch 5/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 50s 314ms/step - apl_penalty: 0.3956 - acc: 0.9768 - focal_loss: 0.0016 - loss: 0.0610 - val_loss: 0.0011
Epoch 6/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 50s 314ms/step - apl_penalty: 0.3598 - acc: 0.9589 - focal_loss: 4.5938e-04 - loss: 0.0544 - val_loss: 0.0019
Epoch 7/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 50s 314ms/step - apl_penalty: 0.4183 - acc: 0.9625 - focal_loss: 0.0025 - loss: 0.0653

In [5]:
# ============================================================
# EfficientNetB0 — baseline training + APL fine-tune (reuses loaded data/masks)
# ============================================================
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as prep_enb0
from tensorflow.keras import layers, models

# ---------- Train baseline ----------
def make_plain_dataset(images_uint8, masks, labels_onehot, prep_fn, batch_size=32, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)
    def _map(img, lbl):
        img = prep_fn(tf.cast(img, tf.float32))
        return img, lbl
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_plain = make_plain_dataset(X_train_raw, train_masks, y_train, prep_enb0)
val_ds_plain = make_plain_dataset(X_val_raw, val_masks, y_val, prep_enb0, shuffle=False)

backbone = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
backbone.trainable = False
enb0_model = models.Sequential([
    backbone, layers.GlobalAveragePooling2D(), layers.Dropout(0.4), layers.Dense(4, activation="softmax")
])
enb0_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
print("Training EfficientNetB0 baseline (frozen backbone)...")
enb0_model.fit(train_ds_plain, validation_data=val_ds_plain, epochs=20,
               callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

backbone.trainable = True
for l in backbone.layers[:-30]:
    l.trainable = False
enb0_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
print("Fine-tuning EfficientNetB0 (last 30 layers unfrozen)...")
enb0_model.fit(train_ds_plain, validation_data=val_ds_plain, epochs=10,
               callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

enb0_baseline_path = "/kaggle/working/efficientnetb0_baseline_seed42.keras"
enb0_model.save(enb0_baseline_path)
print(f"Saved EfficientNetB0 baseline to {enb0_baseline_path}")

# ---------- APL fine-tune on EfficientNetB0 ----------
enb0_baseline = tf.keras.models.load_model(enb0_baseline_path, compile=False)
enb0_snapshot_path = "/kaggle/working/efficientnetb0_baseline_snapshot.keras"
enb0_baseline.save(enb0_snapshot_path)  # untouched copy for fair before/after

def make_apl_dataset_enb0(images_uint8, masks, labels_onehot, batch_size=32, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((images_uint8, masks, labels_onehot))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images_uint8), reshuffle_each_iteration=True)
    def _map(img, mask, lbl):
        img = tf.cast(img, tf.float32)
        if augment:
            img = watermark_masking_augment(img, p=0.5)
        img = prep_enb0(img)
        return (img, mask), lbl
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_apl = make_apl_dataset_enb0(X_train_raw, train_masks, y_train, augment=True)
val_ds_apl = make_apl_dataset_enb0(X_val_raw, val_masks, y_val, shuffle=False, augment=False)

enb0_apl_model = APLModel(enb0_baseline, lambda_apl=0.15)
enb0_apl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), metrics=[tf.keras.metrics.CategoricalAccuracy(name="acc")])
print("\nStarting EfficientNetB0 APL fine-tune...")
enb0_apl_model.fit(train_ds_apl, validation_data=val_ds_apl, epochs=12,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)])
enb0_apl_model.base_model.save("/kaggle/working/efficientnetb0_apl_seed42.keras")

# ---------- Evaluate ----------
enb0_baseline_eval = tf.keras.models.load_model(enb0_snapshot_path, compile=False)

def compute_saliency_and_edgebias_prep(model, images_uint8, masks, prep_fn, batch_size=32):
    n = len(images_uint8)
    all_eb = []
    for i in range(0, n, batch_size):
        bi, bm = images_uint8[i:i+batch_size], masks[i:i+batch_size]
        imgs_prep = prep_fn(tf.cast(bi, tf.float32))
        with tf.GradientTape() as tape:
            tape.watch(imgs_prep)
            out = model(imgs_prep, training=False)
            pred_idx = tf.argmax(out, axis=-1)
            gathered = tf.gather_nd(out, tf.stack([tf.range(tf.shape(out)[0], dtype=tf.int64), pred_idx], axis=1))
        grads = tape.gradient(gathered, imgs_prep)
        saliency = tf.reduce_sum(tf.abs(grads), axis=-1).numpy()
        off_mass = (saliency * (1.0 - bm)).sum(axis=(1, 2))
        total_mass = saliency.sum(axis=(1, 2)) + 1e-8
        all_eb.append(off_mass / total_mass)
    return np.concatenate(all_eb)

notumor_idx = 2
notumor_mask_bool = np.argmax(y_test, axis=1) == notumor_idx
X_notumor = X_test_raw[notumor_mask_bool]
masks_notumor = test_masks[notumor_mask_bool]

eb_before = compute_saliency_and_edgebias_prep(enb0_baseline_eval, X_notumor, masks_notumor, prep_enb0)
eb_after = compute_saliency_and_edgebias_prep(enb0_apl_model.base_model, X_notumor, masks_notumor, prep_enb0)
reduction_pct = 100.0 * (eb_before.mean() - eb_after.mean()) / eb_before.mean()

X_test_prep_enb0 = prep_enb0(tf.cast(X_test_raw, tf.float32))
acc_before = (enb0_baseline_eval.predict(X_test_prep_enb0, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()
acc_after = (enb0_apl_model.base_model.predict(X_test_prep_enb0, batch_size=32, verbose=0).argmax(axis=1) == y_test.argmax(axis=1)).mean()

print(f"\nSanity check (must be False): {np.isclose(eb_before.mean(), eb_after.mean())}")
results_enb0 = {
    "architecture": "EfficientNetB0", "lambda_apl": 0.15,
    "edge_bias_before": float(eb_before.mean()), "edge_bias_after": float(eb_after.mean()),
    "relative_reduction_pct": float(reduction_pct),
    "accuracy_before": float(acc_before), "accuracy_after": float(acc_after),
}
with open("/kaggle/working/pillar1_results_efficientnetb0.json", "w") as f:
    json.dump(results_enb0, f, indent=2)
print("\n=== EFFICIENTNETB0 FINAL RESULTS ===")
print(results_enb0)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training EfficientNetB0 baseline (frozen backbone)...
Epoch 1/20


2026-09-23 09:05:05.793656: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:05.937711: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:06.303559: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:06.445588: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:07.317159: E external/local_xla/xla/stream_

157/158 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5774 - loss: 0.9893

2026-09-23 09:05:24.644165: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:24.784842: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:25.119263: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:25.260951: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:05:26.101374: E external/local_xla/xla/stream_

158/158 ━━━━━━━━━━━━━━━━━━━━ 57s 201ms/step - accuracy: 0.7056 - loss: 0.7449 - val_accuracy: 0.8482 - val_loss: 0.4516
Epoch 2/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.8321 - loss: 0.4546 - val_accuracy: 0.8750 - val_loss: 0.3697
Epoch 3/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.8587 - loss: 0.3908 - val_accuracy: 0.8821 - val_loss: 0.3345
Epoch 4/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.8635 - loss: 0.3681 - val_accuracy: 0.8768 - val_loss: 0.3126
Epoch 5/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.8752 - loss: 0.3384 - val_accuracy: 0.8929 - val_loss: 0.2940
Epoch 6/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.8833 - loss: 0.3239 - val_accuracy: 0.8946 - val_loss: 0.2823
Epoch 7/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.8788 - loss: 0.3163 - val_accuracy: 0.8875 - val_loss: 0.2788
Epoch 8/20
158/158 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.8917 - loss: 0.2959 - val_accuracy: 0.8

2026-09-23 09:10:17.915797: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:10:18.074431: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


157/158 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - apl_penalty: 0.3349 - acc: 0.9048 - focal_loss: 0.0277 - loss: 0.0780

2026-09-23 09:11:02.694618: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:11:02.844161: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-23 09:11:02.985560: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


158/158 ━━━━━━━━━━━━━━━━━━━━ 113s 367ms/step - apl_penalty: 0.3184 - acc: 0.9339 - focal_loss: 0.0454 - loss: 0.0932 - val_loss: 0.0324
Epoch 2/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 20s 122ms/step - apl_penalty: 0.3065 - acc: 0.9518 - focal_loss: 0.0042 - loss: 0.0502 - val_loss: 0.0089
Epoch 3/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 19s 122ms/step - apl_penalty: 0.2910 - acc: 0.9536 - focal_loss: 0.0080 - loss: 0.0517 - val_loss: 0.0187
Epoch 4/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 19s 121ms/step - apl_penalty: 0.2703 - acc: 0.9536 - focal_loss: 2.0290e-04 - loss: 0.0407 - val_loss: 0.0102
Epoch 5/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 19s 120ms/step - apl_penalty: 0.2753 - acc: 0.9536 - focal_loss: 0.0064 - loss: 0.0477 - val_loss: 0.0100
Epoch 6/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 19s 120ms/step - apl_penalty: 0.2596 - acc: 0.9607 - focal_loss: 0.0062 - loss: 0.0452 - val_loss: 0.0134
Epoch 7/12
158/158 ━━━━━━━━━━━━━━━━━━━━ 19s 122ms/step - apl_penalty: 0.2996 - acc: 0.9607 - focal_loss: 0.0084 - loss: 0.0533 - v